# FraudGuard: Cross-Dataset Financial Fraud Detection and Risk Analytics
## Dataset A: IEEE-CIS Fraud Detection Pipeline
**Course**: CSET371 — Big Data Analytics and Business Intelligence  
**Institution**: Bennett University, Greater Noida  
**Team**: Harshit Pahuja, Rachit Nain, Man Mohan Gaur, Kunal Sehrawat, Bhumi Jindal  

---
### Pipeline Overview
1. Ingestion of high-volume transaction & identity records into Spark DataFrames.
2. Schema validation, null profiling, and left-join integrity checks on `TransactionID`.
3. Leakage-free, time-aware forward splitting (70% Train, 15% Validation, 15% Test).
4. Spark ML feature transformation (`StringIndexer`, `OneHotEncoder`, `Imputer`, `VectorAssembler`).
5. Systematic training and evaluation of 5 Spark ML Classifiers:
   - Logistic Regression
   - Decision Tree Classifier
   - Random Forest Classifier
   - Gradient-Boosted Trees (GBTClassifier)
   - Linear Support Vector Machine (LinearSVC)
6. Decision-threshold risk analysis (False Positives vs. False Negatives cost trade-offs).
7. Export of verified metrics and analytical summaries for Power BI Desktop.

In [ ]:
# Step 0: Dependency Setup (Run this cell if running on Google Colab)
try:
    import pyspark
except ImportError:
    !pip install -q pyspark pyyaml

import os
import sys
from pathlib import Path

# Ensure project root is in sys.path
cwd = Path.cwd()
if str(cwd) not in sys.path:
    sys.path.append(str(cwd))
if str(cwd.parent) not in sys.path:
    sys.path.append(str(cwd.parent))

print("Environment initialized.")

In [ ]:
# Step 1: Initialize Apache Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("FraudGuard-IEEE-CIS")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.shuffle.partitions", "16")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Active SparkSession: v{spark.version}")

### Step 2: Data Ingestion & Schema Inspection
Transactions are stored in `train_transaction.csv` and identity telemetry in `train_identity.csv`.

In [ ]:
from pyspark.sql import functions as F

data_dir = Path("data/raw/ieee_cis")
trans_path = data_dir / "train_transaction.csv"
ident_path = data_dir / "train_identity.csv"

if trans_path.exists() and ident_path.exists():
    print("Loading full IEEE-CIS raw CSVs into Spark DataFrames...")
    trans_df = spark.read.csv(str(trans_path), header=True, inferSchema=True)
    ident_df = spark.read.csv(str(ident_path), header=True, inferSchema=True)
else:
    print("Raw CSVs not detected locally. Generating mock sample for demonstration...")
    from src.data.mock_generator import generate_mock_ieee_cis_data
    mock_df = generate_mock_ieee_cis_data(spark, num_records=1000, fraud_rate=0.035)
    trans_df = mock_df
    ident_df = None

print(f"Transaction count: {trans_df.count()}, Columns: {len(trans_df.columns)}")
if ident_df is not None:
    print(f"Identity count: {ident_df.count()}, Columns: {len(ident_df.columns)}")

### Step 3: Join Validation & Target Label Distribution

In [ ]:
from src.data.loader import perform_ieee_cis_join, validate_target_distribution

if ident_df is not None:
    full_df = perform_ieee_cis_join(trans_df, ident_df, join_key="TransactionID")
else:
    full_df = trans_df

target_stats = validate_target_distribution(full_df, target_col="isFraud")
print("Target Stats:", target_stats)

### Step 4: Leakage-Free Time-Aware Data Splitting
We use `TransactionDT` to split chronologically: first 70% for training, 15% for validation, 15% for testing.

In [ ]:
from src.data.loader import time_aware_split

train_df, val_df, test_df = time_aware_split(
    full_df, time_col="TransactionDT", train_ratio=0.70, val_ratio=0.15, test_ratio=0.15
)
print(f"Train set: {train_df.count()} records")
print(f"Validation set: {val_df.count()} records")
print(f"Test set: {test_df.count()} records")

### Step 5: Feature Engineering Pipeline (Spark ML)
Stages are fitted strictly on `train_df` to avoid data leakage.

In [ ]:
from src.features.engineering import identify_column_types, build_spark_ml_feature_pipeline

num_cols, cat_cols, dropped_cols = identify_column_types(train_df, target_col="isFraud")
feature_pipeline = build_spark_ml_feature_pipeline(num_cols, cat_cols, apply_scaling=True)

# Fit strictly on training split
pipeline_model = feature_pipeline.fit(train_df)

# Transform splits
train_prepared = pipeline_model.transform(train_df).withColumnRenamed("isFraud", "label")
val_prepared = pipeline_model.transform(val_df).withColumnRenamed("isFraud", "label")
test_prepared = pipeline_model.transform(test_df).withColumnRenamed("isFraud", "label")

train_prepared.select("TransactionID", "features", "label").show(5, truncate=False)

### Step 6: Systematic Training & Evaluation of the 5 Classifiers

In [ ]:
from src.models.trainer import train_model
from src.evaluation.metrics import evaluate_predictions
import pandas as pd

models_to_train = ["logistic_regression", "decision_tree", "random_forest", "gbt", "linear_svc"]
evaluation_records = []
trained_models = {}

for model_name in models_to_train:
    print(f"\nTraining {model_name}...")
    model, duration = train_model(model_name, train_prepared)
    trained_models[model_name] = model
    
    # Held-out test evaluation
    preds = model.transform(test_prepared)
    metrics = evaluate_predictions(preds, model_name=model_name)
    metrics["training_time_sec"] = duration
    evaluation_records.append(metrics)

results_df = pd.DataFrame(evaluation_records)
display(results_df)
results_df.to_csv("outputs/metrics/model_comparison_ieee_cis.csv", index=False)
results_df.to_csv("powerbi/data/model_comparison.csv", index=False)
print("Metrics exported successfully.")

### Step 7: Decision Threshold & Operational Risk Analysis
Simulate financial losses ($500 per FN) vs. operational alert review friction ($10 per FP).

In [ ]:
from src.risk.thresholds import scan_decision_thresholds

# Use calibrated probability predictions from Random Forest or Logistic Regression
rf_preds = trained_models["random_forest"].transform(test_prepared)
threshold_results = scan_decision_thresholds(
    rf_preds, min_thresh=0.05, max_thresh=0.95, step=0.05, cost_fp=10.0, cost_fn=500.0
)

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df.head(10))
threshold_df.to_csv("outputs/summaries/threshold_analysis_ieee_cis.csv", index=False)
threshold_df.to_csv("powerbi/data/threshold_analysis.csv", index=False)
print("Threshold analysis exported successfully for Power BI.")